# Module 21: Testing & Debugging — Exercises

Practice writing tests, mocking, debugging, and setting up CI for ML pipelines.
Each exercise builds on the previous one.

### Exercise 1: Write a Basic pytest Test

Write a function `standardize(series)` that returns z-scores of a pandas Series.
Then write a pytest test that:
- Verifies the mean of the result is approximately 0
- Verifies the std of the result is approximately 1
- Handles the case where all values are identical

In [ ]:
# Your code here
import pandas as pd
import numpy as np


def standardize(series):
    pass


# Write your tests below

### Exercise 2: Parametrized Tests

Write a function `one_hot_encode(df, column)` that one-hot encodes a categorical column.
Write a parametrized test with at least 3 cases:
- Column with 3 categories
- Column with 2 categories
- Column with 1 category (edge case)

In [ ]:
# Your code here


def one_hot_encode(df, column):
    pass


# Write parametrized test

### Exercise 3: Fixtures and conftest

Create a fixture that generates a DataFrame with:
- 100 rows of random data
- Columns: "feature_1", "feature_2", "target"
- "target" is linearly related to the features plus noise

Write two tests using this fixture:
- Test that a LinearRegression model achieves R^2 > 0.5 on this data
- Test that the correlation between features and target is non-zero

In [ ]:
# Your code here
import pytest
from sklearn.linear_model import LinearRegression


@pytest.fixture
def synthetic_data():
    pass


# Write your tests

### Exercise 4: Mocking an API Call

Given this function that fetches data from an external API:

```python
def fetch_stock_data(ticker, api_key):
    import requests
    url = f"https://api.example.com/v1/{ticker}?key={api_key}"
    response = requests.get(url)
    return response.json()
```

Use `unittest.mock.patch` to test:
- That the function returns expected data when API responds correctly
- That the function raises an exception when API returns 500
- That the URL is constructed correctly

In [ ]:
# Your code here


def fetch_stock_data(ticker, api_key):
    import requests
    url = f"https://api.example.com/v1/{ticker}?key={api_key}"
    response = requests.get(url)
    return response.json()


# Write tests with mock

### Exercise 5: Debug a Broken Training Loop

The training loop below has a bug. Use pdb (or add logging) to find and fix it.
The expected behavior is that loss decreases monotonically.

In [ ]:
import numpy as np


def broken_training_loop(X, y, lr=0.01, epochs=100):
    n_features = X.shape[1]
    weights = np.zeros(n_features + 1)
    bias_idx = 0  # BUG: this should be the bias term position

    for epoch in range(epochs):
        preds = X @ weights[1:] + weights[bias_idx]
        errors = preds - y
        loss = np.mean(errors ** 2) / 2

        grad_w = (X.T @ errors) / len(y)
        grad_b = np.mean(errors)

        weights[1:] -= lr * grad_w
        weights[bias_idx] -= lr * grad_b

        if epoch == 50:
            print(f"Loss at epoch 50: {loss:.6f}")

    return weights, loss


# Generate test data
np.random.seed(42)
X = np.random.randn(200, 3)
true_w = np.array([0.5, -1.2, 0.8])
true_b = 2.0
y = X @ true_w + true_b + np.random.randn(200) * 0.1

# Run and inspect — does loss decrease?
weights, final_loss = broken_training_loop(X, y)
print(f"Final loss: {final_loss:.6f}")
print(f"True weights: {true_w}")
print(f"True bias: {true_b}")
print(f"Learned weights: {weights[1:]}")
print(f"Learned bias: {weights[0]}")

### Exercise 6: Test Coverage Analysis

Add test coverage reporting for the exercises above.
Then answer:
1. Which lines/ functions have the lowest coverage?
2. What edge cases are missing from your tests?
3. How would you improve coverage to 90%+?

In [ ]:
# Run coverage
print("Run in terminal:")
print("  pytest exercises.ipynb --cov=. --cov-report=term-missing -v")
print()
print("Write your analysis below:")

### Exercise 7: Test an ML Pipeline

Write tests for this pipeline function. Test each component separately
using mocking for expensive operations.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


def run_classification_pipeline(train_path, test_path):
    import pandas as pd
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    X_train = train_df.drop(columns=["label"])
    y_train = train_df["label"]
    X_test = test_df.drop(columns=["label"])
    y_test = test_df["label"]

    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(y_train)
    y_test_enc = encoder.transform(y_test)

    model = RandomForestClassifier(n_estimators=100)
    model.fit(X_train, y_train_enc)

    accuracy = (model.predict(X_test) == y_test_enc).mean()
    return model, accuracy


# Write tests that mock pd.read_csv and RandomForestClassifier

### Exercise 8: GitHub Actions CI Configuration

Write a complete GitHub Actions workflow that:
1. Runs on push and pull_request to main
2. Tests on Python 3.9, 3.10, and 3.11
3. Installs dependencies from requirements.txt
4. Runs pytest with coverage (minimum 80%)
5. Uploads coverage report as an artifact
6. Has a separate job for linting with flake8

In [ ]:
# Write your YAML configuration here as a string
ci_yaml = """
"""
print(ci_yaml)

### Exercise 9: TDD — Build a Feature Transformer

Use TDD to build a `DateFeatureExtractor` class:

1. Write tests first (RED):
   - Test that it extracts year, month, day, dayofweek from a date column
   - Test that it handles datetime strings (not just datetime objects)
   - Test that it raises ValueError for invalid date strings
2. Implement the class (GREEN)
3. Refactor to use pd.to_datetime with errors='coerce' (REFACTOR)

In [ ]:
# Your TDD implementation here


class DateFeatureExtractor:
    pass


# Write tests first

### Exercise 10: Debugging with Logging

Add strategic logging to the pipeline below to identify why it fails on certain inputs.
The pipeline divides by certain columns — find which column causes division by zero.

In [ ]:
import logging


def risky_pipeline(df):
    df = df.copy()
    df["ratio_1"] = df["a"] / df["b"]
    df["ratio_2"] = df["c"] / df["d"]
    df["ratio_3"] = df["e"] / df["f"]
    df["score"] = df[["ratio_1", "ratio_2", "ratio_3"]].sum(axis=1)
    return df


# This data causes a problem
bad_data = pd.DataFrame({
    "a": [10, 20, 30],
    "b": [2, 0, 5],
    "c": [1, 2, 3],
    "d": [1, 1, 1],
    "e": [5, 10, 15],
    "f": [1, 2, 0],
})

try:
    result = risky_pipeline(bad_data)
    print(result)
except Exception as e:
    print(f"Error: {e}")


# Add logging to identify which column causes the issue